# 05 — Demand Pattern Analysis

Analyze intermittency, weekday effects, monthly effects, volatility and high/low-demand segments.

In [ ]:

from pathlib import Path
import os, sys
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Project root:", ROOT)


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, ipywidgets as widgets
df=pd.read_parquet(ROOT/"data/interim/cleaned_sales.parquet"); df["weekday_num"]=df.date.dt.dayofweek; df["month_num"]=df.date.dt.month
weekday=df.groupby("weekday_num",observed=True).demand.mean(); display(weekday.to_frame("mean_demand"))
month=df.groupby("month_num",observed=True).demand.mean(); display(month.to_frame("mean_demand"))

In [ ]:
fig,ax=plt.subplots(figsize=(8,4)); weekday.plot(kind="bar",ax=ax); ax.set_title("Average demand by weekday"); plt.show(); fig,ax=plt.subplots(figsize=(8,4)); month.plot(kind="bar",ax=ax); ax.set_title("Average demand by month"); plt.show()

In [ ]:
summary=df.groupby("id",observed=True).demand.agg(mean="mean",std="std",zero_rate=lambda s:(s==0).mean(),total="sum"); summary["cv"]=summary["std"]/summary["mean"].replace(0,np.nan); print(summary.describe().T)

In [ ]:
def series_view(series_id):
    q=df[df.id.astype(str)==str(series_id)].sort_values("date"); fig,ax=plt.subplots(figsize=(10,4)); q.set_index("date").demand.tail(120).plot(ax=ax); ax.set_title(f"Recent demand: {series_id}"); plt.show(); print(q.demand.describe())
ids=sorted(df.id.astype(str).unique()); sel=widgets.Dropdown(options=ids,description="Series"); widgets.interact(series_view,series_id=sel)